# **Templates - Making Prompts Dynamic and Reusable**
1. Input to the Chat Model
    - Types of Concrete Messages - System, Human and AI
    - Types of Templates - System, Human and AI
2. General Purpose Prompt Template
    - Creating a PromptTemplate
    - Passing Placeholder Values to PromptTemplate
    - Convert StringPromptValue to String/Messages
3. ChatPromptTemplate
    - Creating a ChatPromptTemplate
    - Passing Placeholder Values to ChatPromptTemplate
    - Convert ChatPromptValue to String/Messages
4. MessagePlaceholder

## **Input to the Chat Model**

Note that the input to the chat model can be:
1. Individual Message
2. List of Message

### **Example Template and Concrete Message**
**Input:** user_name="Kanav" -> **Template:** "Hi, My name is {user_name}" -> **Concrete Message:** "Hi, My name is Kanav"

### **Types of Concrete Messages**

These are actual, concrete message objects that contain the final, filled-in text content and their respective roles (system, human, AI). They represent a single turn or instruction in a conversation. Example:
- SystemMessage
- HumanMessage
- AIMessage

They are the "data" that the LLM processes.

**They do not contain placeholders ({}). Their content is fully resolved.**

In [1]:
# ! pip install langchain

In [2]:
from langchain_core.messages import SystemMessage

system_instruction = SystemMessage(content="You are a polite and friendly chatbot.")

system_instruction

SystemMessage(content='You are a polite and friendly chatbot.', additional_kwargs={}, response_metadata={})

In [3]:
from langchain_core.messages import HumanMessage

user_query = HumanMessage(content="What is the weather like today?")

user_query

HumanMessage(content='What is the weather like today?', additional_kwargs={}, response_metadata={})

In [4]:
from langchain_core.messages import AIMessage

ai_response = AIMessage(content="I'm sorry, I don't have access to real-time weather information.")

ai_response

AIMessage(content="I'm sorry, I don't have access to real-time weather information.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])

### **Types of Templates**

These are classes that represent templates for specific types of messages (System, Human, AI). They define the structure and content of a message, often including placeholders ({variable_name}) that need to be filled in later. Example:
- SystemMessagePromptTemplate
- HumanMessagePromptTemplate
- AIMessagePromptTemplate

They are not directly sent to the LLM. They first need to be "formatted" or "invoked" to produce concrete `*Message` objects.

In [5]:
from langchain_core.prompts import SystemMessagePromptTemplate

system_template = SystemMessagePromptTemplate.from_template(
    "You are a helpful assistant specialized in {topic}."
)

system_template

SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='You are a helpful assistant specialized in {topic}.'), additional_kwargs={})

In [7]:
system_template.input_variables

['topic']

In [6]:
from langchain_core.prompts import HumanMessagePromptTemplate

human_template = HumanMessagePromptTemplate.from_template(
    "My name is {user_name}. Can you explain {concept}?", 
)

human_template

HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['concept', 'user_name'], input_types={}, partial_variables={}, template='My name is {user_name}. Can you explain {concept}?'), additional_kwargs={})

In [8]:
human_template.input_variables

['concept', 'user_name']

## **General Purpose Prompt Template**

### **Creating a PromptTemplate**
1. Using .from_template()
2. Using direct init

In [9]:
from langchain_core.prompts import PromptTemplate

# Using .from_template()
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)

# Usig direct init
prompt_template = PromptTemplate(
    template="Tell me a {adjective} joke about {content}."
)


prompt_template

PromptTemplate(input_variables=['adjective', 'content'], input_types={}, partial_variables={}, template='Tell me a {adjective} joke about {content}.')

In [10]:
prompt_template.input_variables

['adjective', 'content']

### **Passing Placeholder Values to Prompt Template**
1. Using .format()
2. Using .format_prompt()
3. Using .invoke()

In [12]:
# Important Note: This returns a string
prompt_template.format(adjective="funny", content="genai")

'Tell me a funny joke about genai.'

In [13]:
# Important Note: This returns a StringPromptValue
prompt_template.format_prompt(adjective="funny", content="genai")

StringPromptValue(text='Tell me a funny joke about genai.')

In [14]:
# Important Note: This returns a StringPromptValue
prompt_template.invoke({"adjective":"funny", "content":"genai"})

StringPromptValue(text='Tell me a funny joke about genai.')

### **Convert StringPromptValue to String or List of Messages**
1. .to_string()
2. .to_messages()

In [16]:
spv = prompt_template.invoke({"adjective":"funny", "content":"genai"})

spv.to_string()

'Tell me a funny joke about genai.'

In [17]:
spv.to_messages()

[HumanMessage(content='Tell me a funny joke about genai.', additional_kwargs={}, response_metadata={})]

### **Passing Partial Variables in PromptTemplate**
1. Passing Partial Variable while creating the Prompt Template Object - Using partial_variable argument
2. Passing Partial Variable in an existing template - Using .partial()

In [18]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}.",
    partial_variables={"adjective": "funny"}
)

prompt_template.input_variables

['content']

In [19]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    template="Tell me a {adjective} joke about {content}."
)

prompt_template_pv = prompt_template.partial(adjective="funny")

prompt_template_pv.input_variables

['content']

## **List of Messages using ChatPromptTemplate**

### **Creating a ChatPromptTemplate**
1. Using .from_template()
2. Using .from_messages()
3. Using direct init

In [20]:
from langchain_core.prompts import ChatPromptTemplate

# Using .from_template()
chat_template = ChatPromptTemplate.from_template(
    "Tell me about {topic_name}?"
)

# Using .from_messages()
chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI bot. Your name is {bot_name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "Tell me about {topic_name}."),
    ]
)

# Using direct init
chat_template = ChatPromptTemplate(
    messages=[
        ("system", "You are a helpful AI bot. Your name is {bot_name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "Tell me about {topic_name}."),
    ]
)

chat_template

ChatPromptTemplate(input_variables=['bot_name', 'topic_name'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['bot_name'], input_types={}, partial_variables={}, template='You are a helpful AI bot. Your name is {bot_name}.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Hello, how are you doing?'), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="I'm doing well, thanks!"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic_name'], input_types={}, partial_variables={}, template='Tell me about {topic_name}.'), additional_kwargs={})])

In [21]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate(
    messages=[
        SystemMessagePromptTemplate.from_template("You are a helpful AI bot. Your name is {bot_name}."),
        HumanMessage(content="Hello, how are you doing?"),
        AIMessage(content="I'm doing well, thanks!"),
        HumanMessagePromptTemplate.from_template("Tell me about {topic_name}.")
    ]
)

In [22]:
chat_template.input_variables

['bot_name', 'topic_name']

In [24]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate, AIMessagePromptTemplate
from langchain_core.messages import HumanMessage, AIMessage

# Option A: Using message objects explicitly
chat_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are a helpful AI bot. Your name is {bot_name}.", 
                                                  partial_variables={"bot_name": "ThatAIGuy"}),
        HumanMessage(content="Hello, how are you doing?"),
        AIMessage(content="I'm doing well, thanks!"),
        HumanMessagePromptTemplate.from_template("Tell me about {topic_name}."),
    ]
)

# chat_template.input_variables

### **Passing Placeholder Values to ChatPromptTemplate**
1. Using .format()
2. Using .format_prompt()
3. Using .format_messages()
4. Using .invoke()

In [25]:
# Important Note: This returns a string

chat_template.format(bot_name="Alice", topic_name="langchain")

"System: You are a helpful AI bot. Your name is Alice.\nHuman: Hello, how are you doing?\nAI: I'm doing well, thanks!\nHuman: Tell me about langchain."

In [26]:
# Important Note: This returns a ChatPromptValue

chat_template.format_prompt(bot_name="Alice", topic_name="langchain")

ChatPromptValue(messages=[SystemMessage(content='You are a helpful AI bot. Your name is Alice.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Tell me about langchain.', additional_kwargs={}, response_metadata={})])

In [27]:
# Important Note: This returns a List of messages

chat_template.format_messages(bot_name="Alice", topic_name="langchain")

[SystemMessage(content='You are a helpful AI bot. Your name is Alice.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Tell me about langchain.', additional_kwargs={}, response_metadata={})]

In [28]:
# Important Note: This returns a ChatPromptValue

chat_template.invoke({"bot_name":"Alice", "topic_name":"langchain"})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful AI bot. Your name is Alice.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Tell me about langchain.', additional_kwargs={}, response_metadata={})])

### **Convert ChatPromptValue to String or List of Messages**
1. .to_string()
2. .to_messages()

In [29]:
cpv = chat_template.invoke({"bot_name":"Alice", "topic_name":"langchain"})

In [30]:
cpv.to_string()

"System: You are a helpful AI bot. Your name is Alice.\nHuman: Hello, how are you doing?\nAI: I'm doing well, thanks!\nHuman: Tell me about langchain."

In [31]:
cpv.to_messages()

[SystemMessage(content='You are a helpful AI bot. Your name is Alice.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Tell me about langchain.', additional_kwargs={}, response_metadata={})]

### **Passing Partial Variables in ChatPromptTemplate**
1. Passing Partial Variable while creating the ChatPromptTemplate Object - Using partial_variable argument
2. Passing Partial Variable in an existing template - Using .partial()

In [32]:
# Passing Partial Variable while creating the ChatPromptTemplate Object - Using partial_variable argument

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate(
    messages=[
        ("system", "You are a helpful AI bot. Your name is {bot_name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "Tell me about {topic_name}."),
    ],
    partial_variables={"bot_name": "Alice"}
)

chat_template.input_variables

['topic_name']

In [33]:
chat_template.invoke({"topic_name": "langchain"}).to_messages()

[SystemMessage(content='You are a helpful AI bot. Your name is Alice.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Tell me about langchain.', additional_kwargs={}, response_metadata={})]

In [34]:
# Passing Partial Variable in an existing template - Using .partial()

from langchain_core.prompts import ChatPromptTemplate

chat_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI bot. Your name is {bot_name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "Tell me about {topic_name}."),
    ]
)

chat_template_pv = chat_template.partial(bot_name="Tim")

chat_template_pv.input_variables

['topic_name']

In [35]:
chat_template_pv.invoke({"topic_name": "langchain"}).to_messages()

[SystemMessage(content='You are a helpful AI bot. Your name is Tim.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you doing?', additional_kwargs={}, response_metadata={}),
 AIMessage(content="I'm doing well, thanks!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Tell me about langchain.', additional_kwargs={}, response_metadata={})]

### **MessagesPlaceholder**

At its core, `MessagesPlaceholder` is a special type of "placeholder" within a `ChatPromptTemplate` that is designed to accept a sequence of messages (like HumanMessage, AIMessage, SystemMessage).


#### **How it Works?**  
When you include MessagesPlaceholder in your ChatPromptTemplate.from_messages() or ChatPromptTemplate() definition:
1. You give it a variable_name (e.g., "chat_history").
2. When you invoke() the ChatPromptTemplate (or a chain containing it), you must pass a key in your input dictionary that matches this variable_name. The value for this key must be a list of BaseMessage objects.
3. LangChain then takes this list of messages and inserts them directly into the position specified by MessagesPlaceholder within the final list of messages sent to the LLM.

In [36]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Simulate some pre-existing chat history
current_chat_history = [
    HumanMessage(content="What's your favorite color?"),
    AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    HumanMessage(content="Oh, I see. What's your favorite animal then?"),
    AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals."),
]

# Simulate some pre-existing chat history
current_chat_history = [
    ("human", "What's your favorite color?"),
    ("ai", "As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    ("human", "Oh, I see. What's your favorite animal then?"),
    ("ai", "Similarly, I don't have personal experiences to develop preferences for animals."),
]

In [37]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, HumanMessagePromptTemplate

# Define a ChatPromptTemplate that includes MessagesPlaceholder
chat_prompt_with_history = ChatPromptTemplate(
    messages=[
        SystemMessage(content="You are a helpful and knowledgeable assistant."),
        # This is where the magic happens: insert the chat history
        MessagesPlaceholder(variable_name="chat_history"),
        # The new human message for the current turn
        HumanMessagePromptTemplate.from_template("{new_question}"),
    ]
)

chat_prompt_with_history.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{new_question}


In [38]:
try:
    chat_prompt_with_history.invoke({"new_question": "Can you summarize our conversation so far?"})
except:
    print("KeyError: Input to ChatPromptTemplate is missing variables {'chat_history'}.")

KeyError: Input to ChatPromptTemplate is missing variables {'chat_history'}.


In [39]:
formatted_messages = chat_prompt_with_history.invoke(
    {
        "new_question": "Can you summarize our conversation so far?", 
        "chat_history": current_chat_history
    }
)

formatted_messages.to_messages()

[SystemMessage(content='You are a helpful and knowledgeable assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's your favorite color?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content="Oh, I see. What's your favorite animal then?", additional_kwargs={}, response_metadata={}),
 AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Can you summarize our conversation so far?', additional_kwargs={}, response_metadata={})]

In [40]:
for msg in formatted_messages.to_messages():
    msg.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.
================================ Human Message =================================

What's your favorite color?
================================== Ai Message ==================================

As an AI, I don't have feelings or preferences, so I don't have a favorite color.
================================ Human Message =================================

Oh, I see. What's your favorite animal then?
================================== Ai Message ==================================

Similarly, I don't have personal experiences to develop preferences for animals.
================================ Human Message =================================

Can you summarize our conversation so far?


#### **Making MessagesPlaceholder Optional**

In [41]:
# Define a ChatPromptTemplate that includes MessagesPlaceholder
chat_prompt_with_history = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a helpful and knowledgeable assistant."),
        # This is where the magic happens: insert the chat history
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        # The new human message for the current turn
        HumanMessagePromptTemplate.from_template("{new_question}"),
    ]
)

chat_prompt_with_history.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.

============================= Messages Placeholder =============================

{chat_history}

================================ Human Message =================================

{new_question}


#### **Limiting the number of messages**

In [42]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Simulate some pre-existing chat history
current_chat_history = [
    HumanMessage(content="What's your favorite color?"),
    AIMessage(content="As an AI, I don't have feelings or preferences, so I don't have a favorite color."),
    HumanMessage(content="Oh, I see. What's your favorite animal then?"),
    AIMessage(content="Similarly, I don't have personal experiences to develop preferences for animals."),
]

# Define a ChatPromptTemplate that includes MessagesPlaceholder
chat_prompt_with_history = ChatPromptTemplate(
    messages = [
        SystemMessage(content="You are a helpful and knowledgeable assistant."),
        # This is where the magic happens: insert the chat history
        MessagesPlaceholder(variable_name="chat_history", optional=True, n_messages=2),
        # The new human message for the current turn
        HumanMessagePromptTemplate.from_template("{new_question}"),
    ] 
)

formatted_messages = chat_prompt_with_history.invoke(
    {
        "chat_history": current_chat_history,
        "new_question": "Can you summarize our conversation so far?"
    }
)

for msg in formatted_messages.to_messages():
    msg.pretty_print()

================================ System Message ================================

You are a helpful and knowledgeable assistant.
================================ Human Message =================================

Oh, I see. What's your favorite animal then?
================================== Ai Message ==================================

Similarly, I don't have personal experiences to develop preferences for animals.
================================ Human Message =================================

Can you summarize our conversation so far?
